In [1]:
import psycopg2
import numpy as np
from sentence_transformers import SentenceTransformer
import requests

Load Embedding Model

In [19]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Connect to PostgreSQ

In [20]:
conn = psycopg2.connect(
    host="localhost",
    database="postgres",
    user="postgres",
    password="Hybye321$",
    port="5432"
)

cursor = conn.cursor()

User Complaint Input

In [21]:
query = input("Enter your query: ")

Generate Query Embedding

In [22]:
query_vector = model.encode(query)

vector_str = "[" + ",".join(map(str, query_vector)) + "]"

Generate Query Embedding

In [23]:
conn.rollback()

cursor.execute(
"""
SELECT text, label
FROM complaints
ORDER BY embedding <-> %s::vector
LIMIT 3
""",
(vector_str,)
)

results = cursor.fetchall()

Build Context for the LLM

In [24]:
context = ""

for i, r in enumerate(results):
    context += f"""
Similar Case {i+1}:
Complaint: {r[0]}
Label: {r[1]}
"""

Create Prompt for the LLM

In [25]:
prompt = f"""
You are a financial dispute analysis assistant.

Classify the complaint into one of:
- dispute
- not_dispute
- escalate_to_human

User complaint:
{query}

Similar past complaints:
{context}

Return JSON:
{{"label":"", "reasoning":""}}
"""

Send Prompt to Phi-3 (Ollama)

In [26]:
response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "phi3",
        "prompt": prompt,
        "stream": False
    }
)

print(response.json()["response"])

```json
{
  "label": "dispute",
  "reasoning": "The user's complaint involves a charge that they allege was unauthorized and not resolved by the credit card company. This situation aligns with common disputes where a charge is contested due to non-recognition or lack of transaction records. The user has already reported the unauthorized charge and seeks resolution, which typically begins with a dispute process before escalation. Therefore, the appropriate classification for this complaint is 'dispute'."
}
```
